In [1]:
import torch
import matplotlib.pyplot as plt
from torch import nn
print("Gpu enabled: ", torch.cuda.is_available())

Gpu enabled:  True


In [2]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Generate 500 samples
num_samples = 500

engine_size = torch.empty(num_samples, 1).uniform_(1.0, 5.0)
horsepower = torch.empty(num_samples, 1).uniform_(80.0, 450.0)
age = torch.empty(num_samples, 1).uniform_(0.0, 15.0)
mileage = torch.empty(num_samples, 1).uniform_(5000.0, 150000.0)

# Combine into a single feature matrix X of shape (500, 4)
X_raw = torch.cat([engine_size, horsepower, age, mileage], dim=1)

# True coefficients
# Price = 2500*Engine + 150*HP - 900*Age - 0.12*Mileage + 12000 + Noise
true_weights = torch.tensor([[2500.0], [150.0], [-900.0], [-0.12]])
true_bias = 12000.0
noise = torch.randn(num_samples, 1) * 1500.0

y_raw = torch.matmul(X_raw, true_weights) + true_bias + noise

print("X shape:", X_raw.shape)  # Expected: [500, 4]
print("y shape:", y_raw.shape)  # Expected: [500, 1]
print("\nFirst 3 rows of X_raw:\n", X_raw[:3])
print("\nFirst 3 rows of y_raw:\n", y_raw[:3])

X shape: torch.Size([500, 4])
y shape: torch.Size([500, 1])

First 3 rows of X_raw:
 tensor([[4.5291e+00, 2.2020e+02, 1.0941e+01, 1.3926e+05],
        [4.6600e+00, 1.7562e+02, 4.0933e+00, 7.3324e+04],
        [2.5315e+00, 2.9646e+02, 3.6101e+00, 9.1568e+04]])

First 3 rows of y_raw:
 tensor([[28305.9453],
        [37126.8359],
        [47122.1016]])


In [4]:
split_percentage = int(0.8* len(X_raw))

x_train, y_train = X_raw[:split_percentage], y_raw[:split_percentage]
x_test, y_test = X_raw[split_percentage:], y_raw[split_percentage:]

len(x_train), len(y_train), len(x_test), len(y_test)

(400, 400, 100, 100)

In [ ]:
def plot_graph(x_train=x_train, y_train=y_train, x_test=x_test, y_test=y_test, predictions=None):
    plt.figure(figsize=(10, 6))
    
    # Convert PyTorch tensors to CPU/numpy for plotting
    def to_np(tensor):
        return tensor.detach().cpu().numpy() if isinstance(tensor, torch.Tensor) else tensor

    y_test_np = to_np(y_test)

    if predictions is not None:
        preds_np = to_np(predictions)
        # Scatter plot: Actual Price (X-axis) vs Predicted Price (Y-axis)
        plt.scatter(y_test_np, preds_np, c="g", s=25, alpha=0.7, label="Model Predictions")
        
        # 45-degree ideal line where Actual == Predicted
        min_val = min(y_test_np.min(), preds_np.min())
        max_val = max(y_test_np.max(), preds_np.max())
        plt.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2, label="Ideal Line (Perfect Match)")
        
        plt.xlabel("Actual Car Price ($)", fontsize=12)
        plt.ylabel("Predicted Car Price ($)", fontsize=12)
        plt.title("Actual vs Predicted Car Prices", fontsize=14)
    else:
        # Plot distributions of train and test prices
        y_train_np = to_np(y_train)
        plt.scatter(range(len(y_train_np)), y_train_np, c="b", s=20, label="Training Prices")
        plt.scatter(range(len(y_train_np), len(y_train_np) + len(y_test_np)), y_test_np, c="r", s=20, label="Test Prices")
        plt.xlabel("Sample Index", fontsize=12)
        plt.ylabel("Car Price ($)", fontsize=12)
        plt.title("Car Price Dataset Distribution", fontsize=14)

    plt.legend(prop={"size": 12})
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()

# You can call it exactly as before:
plot_graph()
# Or with predictions:
# plot_graph(predictions=y_preds)
